# DocuMed Detector — ABP Procesamiento de Imágenes

Notebook local para ejecutar en **Visual Studio Code**.

Este notebook asume que las carpetas del dataset están en la misma carpeta del notebook:

```text
IMAGENES/
├── train/
│   ├── _annotations.csv
│   └── *.jpg
├── valid/
│   ├── _annotations.csv
│   └── *.jpg
├── test/
│   ├── _annotations.csv
│   └── *.jpg
└── DocuMed_Detector_ABP_Local_VSCode.ipynb
```

Formato esperado de anotaciones RetinaNet:

```text
filename,xmin,ymin,xmax,ymax,class
```


## 1. Instalar dependencias

In [ ]:
# Ejecutar una sola vez si faltan librerías.
# En VS Code/Jupyter también funciona con !pip.

!pip install -q ultralytics opencv-python pandas matplotlib pyyaml

## 2. Importar librerías

In [ ]:
from pathlib import Path
import os
import shutil
import random
import yaml

import cv2
import pandas as pd
import matplotlib.pyplot as plt

from ultralytics import YOLO

random.seed(42)

print("Librerías cargadas correctamente")

## 3. Configurar rutas locales

In [ ]:
# IMPORTANTE:
# En VS Code, Path.cwd() normalmente apunta a la carpeta abierta como workspace.
# En tu caso debería ser la carpeta IMAGENES, donde están train, valid y test.

BASE_DIR = Path.cwd()

# Buscar automáticamente la carpeta raíz del dataset
def buscar_dataset_root(base_dir: Path) -> Path:
    candidatos = [
        base_dir,
        base_dir / "HOSPITAL SAN JUAN BAUTISTA.v2i.retinanet",
        base_dir / "dataset",
        base_dir / "data",
    ]

    for candidato in candidatos:
        if all((candidato / split / "_annotations.csv").exists() for split in ["train", "valid", "test"]):
            return candidato

    # Búsqueda recursiva si el dataset quedó dentro de una subcarpeta
    for candidato in base_dir.rglob("_annotations.csv"):
        posible_root = candidato.parent.parent
        if all((posible_root / split / "_annotations.csv").exists() for split in ["train", "valid", "test"]):
            return posible_root

    raise FileNotFoundError("No se encontraron train/valid/test con _annotations.csv. Revisar ubicación del dataset.")

DATASET_ROOT = buscar_dataset_root(BASE_DIR)

PROJECT_DIR = BASE_DIR / "documed_detector"
YOLO_DIR = PROJECT_DIR / "yolo_dataset"
OUTPUT_DIR = PROJECT_DIR / "outputs"
EDA_DIR = OUTPUT_DIR / "eda"
PRED_DIR = OUTPUT_DIR / "predicciones"

for path in [PROJECT_DIR, YOLO_DIR, OUTPUT_DIR, EDA_DIR, PRED_DIR]:
    path.mkdir(parents=True, exist_ok=True)

SPLITS = ["train", "valid", "test"]
COLUMNS = ["filename", "xmin", "ymin", "xmax", "ymax", "class"]

CLASS_NAMES = [
    "autorizacion",
    "cantidad",
    "codigo",
    "copago",
    "cuota_moderadora",
    "factura",
    "ingreso",
    "paciente",
    "total",
    "valor_unitario",
]

print("Carpeta base:", BASE_DIR)
print("Dataset detectado en:", DATASET_ROOT)
print("Carpeta del proyecto:", PROJECT_DIR)

## 4. Validar imágenes JPG y anotaciones

In [ ]:
for split in SPLITS:
    split_dir = DATASET_ROOT / split
    csv_path = split_dir / "_annotations.csv"
    jpgs = list(split_dir.glob("*.jpg"))

    print(f"{split}:")
    print(f"  CSV existe: {csv_path.exists()}")
    print(f"  Imágenes .jpg: {len(jpgs)}")

## 5. Cargar anotaciones

In [ ]:
def cargar_anotaciones(split: str) -> pd.DataFrame:
    csv_path = DATASET_ROOT / split / "_annotations.csv"
    df_split = pd.read_csv(csv_path, names=COLUMNS)
    df_split["split"] = split
    return df_split

df = pd.concat([cargar_anotaciones(split) for split in SPLITS], ignore_index=True)

# Convertir coordenadas a numéricas por seguridad
for col in ["xmin", "ymin", "xmax", "ymax"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["xmin", "ymin", "xmax", "ymax", "class", "filename"]).copy()

print("Total de anotaciones:", len(df))
print("Total de imágenes únicas:", df["filename"].nunique())
print("Total de clases:", df["class"].nunique())

display(df.head())

## 6. Análisis exploratorio del dataset

In [ ]:
resumen_imagenes = []

for split in SPLITS:
    split_dir = DATASET_ROOT / split
    imagenes = list(split_dir.glob("*.jpg"))
    resumen_imagenes.append({"split": split, "imagenes": len(imagenes)})

resumen_df = pd.DataFrame(resumen_imagenes)
display(resumen_df)

plt.figure(figsize=(6, 4))
plt.bar(resumen_df["split"], resumen_df["imagenes"])
plt.title("Cantidad de imágenes por partición")
plt.xlabel("Partición")
plt.ylabel("Cantidad de imágenes")
plt.grid(axis="y")
plt.show()

In [ ]:
conteo_clases = df["class"].value_counts().sort_values(ascending=False)

display(conteo_clases.reset_index().rename(columns={"index": "clase", "class": "cantidad"}))

plt.figure(figsize=(11, 5))
conteo_clases.plot(kind="bar")
plt.title("Distribución de anotaciones por clase")
plt.xlabel("Clase")
plt.ylabel("Cantidad de anotaciones")
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y")
plt.tight_layout()
plt.savefig(EDA_DIR / "distribucion_clases.png", dpi=150)
plt.show()

In [ ]:
# Medir tamaños reales de imágenes
tamanios = []

for split in SPLITS:
    for img_path in (DATASET_ROOT / split).glob("*.jpg"):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]
        tamanios.append({"split": split, "filename": img_path.name, "width": w, "height": h})

tamanios_df = pd.DataFrame(tamanios)
display(tamanios_df.groupby("split")[["width", "height"]].describe())

plt.figure(figsize=(7, 5))
plt.scatter(tamanios_df["width"], tamanios_df["height"], alpha=0.7)
plt.title("Dimensiones originales de las imágenes")
plt.xlabel("Ancho (px)")
plt.ylabel("Alto (px)")
plt.grid(True)
plt.tight_layout()
plt.savefig(EDA_DIR / "tamanios_imagenes.png", dpi=150)
plt.show()

## 7. Visualizar imagen con anotaciones originales

In [ ]:
def mostrar_imagen_con_anotaciones(split: str, filename: str):
    img_path = DATASET_ROOT / split / filename
    image = cv2.imread(str(img_path))

    if image is None:
        raise FileNotFoundError(f"No se pudo leer la imagen: {img_path}")

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    anns = df[(df["split"] == split) & (df["filename"] == filename)]

    for _, row in anns.iterrows():
        xmin, ymin, xmax, ymax = map(int, [row["xmin"], row["ymin"], row["xmax"], row["ymax"]])
        label = row["class"]

        cv2.rectangle(image, (xmin, ymin), (xmax, ymax), (255, 0, 0), 2)
        cv2.putText(image, label, (xmin, max(ymin - 5, 15)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)

    plt.figure(figsize=(10, 10))
    plt.imshow(image)
    plt.axis("off")
    plt.title(f"{split} / {filename}")
    plt.show()

# Ejemplo
ejemplo_row = df[df["split"] == "train"].iloc[0]
mostrar_imagen_con_anotaciones("train", ejemplo_row["filename"])

## 8. Convertir formato RetinaNet a YOLO

In [ ]:
CLASS_TO_ID = {name: idx for idx, name in enumerate(CLASS_NAMES)}

def convertir_bbox_a_yolo(xmin, ymin, xmax, ymax, img_w, img_h):
    x_center = ((xmin + xmax) / 2) / img_w
    y_center = ((ymin + ymax) / 2) / img_h
    width = (xmax - xmin) / img_w
    height = (ymax - ymin) / img_h
    return x_center, y_center, width, height

def limpiar_y_crear_yolo_dir():
    if YOLO_DIR.exists():
        shutil.rmtree(YOLO_DIR)

    for split in SPLITS:
        (YOLO_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
        (YOLO_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

limpiar_y_crear_yolo_dir()

for split in SPLITS:
    split_df = df[df["split"] == split].copy()
    split_img_dir = DATASET_ROOT / split

    for filename, group in split_df.groupby("filename"):
        img_path = split_img_dir / filename
        img = cv2.imread(str(img_path))

        if img is None:
            print(f"Imagen no leída, se omite: {img_path}")
            continue

        img_h, img_w = img.shape[:2]

        # Copiar imagen JPG
        shutil.copy2(img_path, YOLO_DIR / "images" / split / filename)

        label_path = YOLO_DIR / "labels" / split / filename.replace(".jpg", ".txt")

        lines = []
        for _, row in group.iterrows():
            clase = row["class"]

            if clase not in CLASS_TO_ID:
                print(f"Clase no registrada, se omite: {clase}")
                continue

            xmin, ymin, xmax, ymax = float(row["xmin"]), float(row["ymin"]), float(row["xmax"]), float(row["ymax"])

            # Recorte defensivo para evitar coordenadas fuera de imagen
            xmin = max(0, min(xmin, img_w - 1))
            xmax = max(0, min(xmax, img_w - 1))
            ymin = max(0, min(ymin, img_h - 1))
            ymax = max(0, min(ymax, img_h - 1))

            if xmax <= xmin or ymax <= ymin:
                continue

            x_center, y_center, width, height = convertir_bbox_a_yolo(xmin, ymin, xmax, ymax, img_w, img_h)
            class_id = CLASS_TO_ID[clase]
            lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

        label_path.write_text("\n".join(lines), encoding="utf-8")

print("Conversión a YOLO finalizada en:", YOLO_DIR)

## 9. Crear archivo data.yaml

In [ ]:
data_yaml = {
    "path": str(YOLO_DIR.resolve()),
    "train": "images/train",
    "val": "images/valid",
    "test": "images/test",
    "nc": len(CLASS_NAMES),
    "names": CLASS_NAMES,
}

DATA_YAML_PATH = YOLO_DIR / "data.yaml"

with open(DATA_YAML_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False, allow_unicode=True)

print(DATA_YAML_PATH)
print(DATA_YAML_PATH.read_text(encoding="utf-8"))

## 10. Entrenar modelo YOLOv8

In [ ]:
# Para prueba rápida usar 10 épocas.
# Para entrega final usar 50 o más, dependiendo de la capacidad de tu PC.

EPOCHS = 10
IMG_SIZE = 960
BATCH_SIZE = 2

model = YOLO("yolov8n.pt")

train_results = model.train(
    data=str(DATA_YAML_PATH),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project=str(OUTPUT_DIR),
    name="documed_detector_yolov8n",
    exist_ok=True
)

## 11. Evaluar modelo

In [ ]:
BEST_MODEL_PATH = OUTPUT_DIR / "documed_detector_yolov8n" / "weights" / "best.pt"

if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(f"No se encontró el modelo entrenado: {BEST_MODEL_PATH}")

best_model = YOLO(str(BEST_MODEL_PATH))

metrics = best_model.val(
    data=str(DATA_YAML_PATH),
    split="test",
    imgsz=IMG_SIZE,
    project=str(OUTPUT_DIR),
    name="evaluacion_test",
    exist_ok=True
)

print("Evaluación finalizada")

## 12. Predicción sobre imágenes de prueba

In [ ]:
TEST_IMAGES_DIR = YOLO_DIR / "images" / "test"

results = best_model.predict(
    source=str(TEST_IMAGES_DIR),
    imgsz=IMG_SIZE,
    conf=0.25,
    save=True,
    project=str(PRED_DIR),
    name="test_predictions",
    exist_ok=True
)

print("Predicciones guardadas en:", PRED_DIR / "test_predictions")

In [ ]:
# Mostrar una predicción guardada
pred_output_dir = PRED_DIR / "test_predictions"
pred_images = list(pred_output_dir.glob("*.jpg"))

if pred_images:
    img = cv2.imread(str(pred_images[0]))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(10, 10))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Ejemplo de predicción del modelo")
    plt.show()
else:
    print("No se encontraron imágenes de predicción guardadas.")

## 13. Exportar resumen para documentación

In [ ]:
resumen = {
    "imagenes_train": len(list((DATASET_ROOT / "train").glob("*.jpg"))),
    "imagenes_valid": len(list((DATASET_ROOT / "valid").glob("*.jpg"))),
    "imagenes_test": len(list((DATASET_ROOT / "test").glob("*.jpg"))),
    "anotaciones_totales": int(len(df)),
    "clases": CLASS_NAMES,
}

resumen_path = OUTPUT_DIR / "resumen_dataset.txt"
with open(resumen_path, "w", encoding="utf-8") as f:
    for k, v in resumen.items():
        f.write(f"{k}: {v}\n")

print(resumen_path.read_text(encoding="utf-8"))